# NB13 — Stage-2 training v2 (SPEC T5)

One notebook, four modes (set `RUN` in the next cell). Run them in this order, one Kaggle session each:

| RUN | What | GPU time |
| --- | --- | --- |
| `PROBE` | 200 steps of BASE and of run A: step time, peak memory, projected hours. **No checkpoints.** | ~30 min |
| `BASE` | NB8 recipe on Extra folds A+B (no fusion, consistency, weights or SpecAugment) | from the probe |
| `A` | Recipe v2 on Extra fold A | from the probe |
| `B` | Recipe v2 on Extra fold B | from the probe |

Every 500 steps: greedy decode of Extra-dev, official scorer, keep the best 3 checkpoints by F1.
Each full run aborts after 200 steps if its projection exceeds 11 h. Never touches QuranMB.v2 or IqraEval_Test_GT.

**Run settings:** Accelerator **GPU T4**, Internet *On*. Add data: `iqra-cache`, the NB11 output, the NB12 output
(`sample_weights.parquet`), `quran-phoneme-model` (checkpoint-5000), the NB8 model output (notebook17aed3d165, parity).
Secrets: `HF_TOKEN`.

**After the run:** Save Version (full runs: then attach this notebook's output to NB14 in T6).
Paste back everything printed by the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Mode, environment, parity, inputs ----------------------------------------------------
RUN = "PROBE"                     # "PROBE", then "BASE", "A", "B" (one Kaggle session each)
PROBE_STEPS = 200
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import json, random, time
from dataclasses import replace
import numpy as np
import pandas as pd
import torch, transformers
from transformers import AutoFeatureExtractor
from datasets import load_dataset, load_from_disk
from iqra import data, model as M, parity, train as T
from iqra.config import Config, find_cache_root, find_input
from iqra.vocab import Vocab

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required for the gated Iqra_Extra_IS26") from e
par = parity.verify_environment(DEVICE)          # decision 27
print("version parity: PASS |", par["here"])
cfg = Config()
cfg.paths.cache_root = find_cache_root()
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
WEIGHTS_DIR = find_input("sample_weights.parquet", what="the NB12 output")
CKPT = find_input("checkpoint-5000/config.json", what="quran-phoneme-model (holds checkpoint-5000)")
OUT = f"/kaggle/working/nb13_{RUN.lower()}"
os.makedirs(OUT, exist_ok=True)
vocab = Vocab.load(cfg.paths.vocab)
splits = data.load_splits(f"{SPLITS_DIR}/splits.json")
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)
sample_weights = pd.read_parquet(f"{WEIGHTS_DIR}/sample_weights.parquet")
fe = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")      # as NB2 / NB8 / NB14
report = {"repo_commit": REPO_COMMIT, "run": RUN, "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
          "versions": par["here"], "parity": par["pass"]}
print("run:", RUN, "| out:", OUT)


In [ ]:
# ---- Data: training sources, Extra-dev and drift-slice audio, evaluator ---------------------
train_split = load_from_disk(cfg.paths.train_arrow)["train"]
dev_split = load_from_disk(cfg.paths.train_arrow)["dev"]
tts = load_from_disk(cfg.paths.tts_arrow)
tts = tts["train"] if hasattr(tts, "keys") and "train" in tts.keys() else tts
extra = load_dataset(cfg.datasets.extra, split="train", token=HF_TOKEN)
ex_ids = [str(x) for x in list(extra["id"])]
assert ex_ids == splits["extra"]["ids_by_row"]
ex_mis = dict(zip(ex_ids, [str(x) for x in list(extra["phoneme_mis"])]))
train_labels = [str(x) for x in list(train_split["phoneme_ref"])]
tts_labels = [str(x) for x in list(tts["phoneme_mis"])]
assert sample_weights.groupby("dataset").size().to_dict() == {"iqra_train": len(train_split), "iqra_tts": len(tts)}

def audio_of(ds, row):
    return data.to_16k(*data.decode_audio(ds.data.slice(row, 1).column("audio").to_pylist()[0]))

row_of = {c: r for r, c in enumerate(ex_ids)}
dev_audio = {cid: audio_of(extra, row_of[cid]) for cid in truth_dev["ID"]}
slice_ids, slice_rows = splits["iqra_dev_slice"]["ids"], splits["iqra_dev_slice"]["rows"]
pick = sorted(random.Random(13).sample(range(len(slice_ids)), 250))
dev_phon = list(dev_split["phoneme_ref"])
slice_audio = {slice_ids[k]: audio_of(dev_split, slice_rows[k]) for k in pick}
slice_refs = {slice_ids[k]: str(dev_phon[slice_rows[k]]) for k in pick}
evaluator = T.DevEvaluator(dev_audio, truth_dev, slice_audio, slice_refs, fe, vocab.id2tok, vocab.blank_id,
                           DEVICE, dict(zip(truth_dev["ID"], truth_dev["Reference_phn"])))
print(f"train {len(train_split)} | tts {len(tts)} | extra {len(extra)} | Extra-dev {len(dev_audio)} | drift slice {len(slice_audio)}")

def build(run):
    srcs = T.build_sources(run, train_split, train_labels, tts, tts_labels, extra, ex_ids, ex_mis, splits,
                           sample_weights)
    ds = T.CTCDataset(srcs, fe, vocab.tok2id, vocab.tok2id["<unk>"])
    print(f"{run.name}: {len(ds)} training entries ({', '.join(f'{s.name} {len(s.rows)}' for s in srcs)})")
    return ds


In [ ]:
# ---- Spec checks on the real model in this transformers version ------------------------------
m0, info = M.load_model(CKPT, DEVICE)
assert not any(info.values()), info
feats = {k: v.to(DEVICE) for k, v in fe(dev_audio[truth_dev["ID"].iloc[0]], sampling_rate=16_000, return_tensors="pt").items()}
hs = M.encoder_last_equals_last_layer(m0, **feats)
print("hidden-state index check:", hs)
assert hs["hidden_states[-1]==last_layer"] and hs["n_hidden_states"] == 25
with torch.no_grad():
    plain = m0(**feats).logits
    M.add_layer_fusion(m0)
    fused = m0(**feats).logits
diff = float((plain - fused).abs().max())
print(f"fused vs plain at init: max |logit diff| {diff:.5f}, argmax identical {bool(torch.equal(plain.argmax(-1), fused.argmax(-1)))}")
assert diff < 1e-2
report["spec_checks"] = {"hidden_states": hs, "fused_init_max_logit_diff": diff}
del m0
torch.cuda.empty_cache()


In [ ]:
# ---- Train (or probe) -------------------------------------------------------------------------
def fresh(run):
    m, info = M.load_model(CKPT, DEVICE)
    assert not any(info.values()), info
    active = T.configure(m, run)
    print("active config:", active)
    return m, active

results = {}
if RUN == "PROBE":
    for name in ("BASE", "A"):
        run = T.RUNS[name]
        m, active = fresh(run)
        s = T.train(run, m, T.make_loader(build(run), run), evaluator, f"{OUT}/{name.lower()}", DEVICE,
                    probe_steps=PROBE_STEPS)
        results[name] = {"active": active, "sec_per_step": s["sec_per_step"], "peak_mem_gb": s["peak_mem_gb"],
                         "projected_hours_full_run": s["projected_hours_full_run"],
                         "step0_f1": s["step0_eval"]["f1"], "eval_minutes": s["step0_eval"]["eval_minutes"]}
        print(name, results[name])
        del m
        torch.cuda.empty_cache()
    # step 0 of BASE is the stage-1 model itself: it must reproduce B0 (NB14, F1 0.2400)
    assert abs(results["BASE"]["step0_f1"] - 0.2400) < 1e-4, results["BASE"]["step0_f1"]
else:
    run = T.RUNS[RUN]
    m, active = fresh(run)
    with open(f"{OUT}/run_config.json", "w") as f:
        json.dump({"run": T.run_config_dict(run), "active": active, "repo_commit": REPO_COMMIT}, f, indent=1, default=str)
    s = T.train(run, m, T.make_loader(build(run), run), evaluator, OUT, DEVICE)
    results[RUN] = {"active": active, "hours": s["hours"], "sec_per_step": s["sec_per_step"],
                    "peak_mem_gb": s["peak_mem_gb"], "kept": s["kept"],
                    "evals": [{k: e.get(k) for k in ("step", "f1", "precision", "recall", "per_slice",
                                                     "fusion_last_weight")} for e in s["evals"]]}
report["results"] = results
with open(f"{OUT}/nb13_report.json", "w") as f:
    json.dump(report, f, indent=1, default=str)


In [ ]:
# ---- PASTE BACK -------------------------------------------------------------------------------
print(json.dumps(report, indent=1, default=str))
print("files:", sorted(os.listdir(OUT)))
